# 엔티티 메모리 — Store + Tool 패턴

## 이번 노트북 학습 목표

- 대화에서 인물·역할·프로젝트 정보를 추출하는 **엔티티 추출 체인(entity extraction chain)**을 만들어요.
- 추출한 정보를 세션별로 저장하는 **엔티티 스토어(entity store)** 패턴을 익혀요.
- 스토어의 정보를 프롬프트에 주입해 **개인화된 응답**을 만드는 방법을 이해해요.
- 구식 `ConversationEntityMemory` / `ConversationKGMemory`와의 차이를 비교해요.

## 사전 지식

- Modern 메모리 패턴 기초 (`ChatMessageHistory`, `RunnableWithMessageHistory`) (10번 노트북)
- 요약 기반 메모리 패턴 (13번 노트북)

## 구식 vs Modern 메모리 패턴 비교

| 항목 | 구식 (04번, `ConversationEntityMemory`) | Modern 엔티티 메모리 |
|------|----------------------------------------|---------------------|
| 메모리 클래스 | `ConversationEntityMemory(llm=...)` | 없음 (추출 체인 + 스토어로 대체) |
| 엔티티 추출 | 클래스 내부 자동 처리 | `entity_extract_chain` 함수로 명시적 처리 |
| 저장소 | 클래스 내부 dict | 세션별 외부 dict / DB로 분리 |
| LangChain 권장 | 0.2.x 이후 deprecated | 1.0.x 권장 방식 |

> 🎯 **강의 포인트**: 이 노트북이 Ch04 Memory 시리즈의 마지막입니다. Legacy(04번)의 `ConversationEntityMemory` 한 줄이 Modern에서는 추출 체인 + 스토어 + 포맷 함수로 분리된다는 점을 정리해 주세요. 복잡해 보이지만, 각 컴포넌트를 독립적으로 교체·확장할 수 있다는 것이 장점입니다.

> 💡 **실무 팁**: 엔티티 추출마다 LLM 호출이 발생합니다. 모든 발화에서 추출하지 않고, "인물/역할 관련 키워드가 포함된 발화"에만 선택적으로 추출하면 비용을 크게 줄일 수 있습니다.

## 메모리 진화 다이어그램

```mermaid
flowchart LR
    subgraph OLD["구식 (04~06번)"]
        direction TB
        E1[ConversationEntityMemory]
        E2[ConversationKGMemory]
        E3[ConversationSummaryMemory]
    end

    subgraph NEW["Modern (14번)"]
        direction TB
        N1[엔티티 추출 체인<br/>entity_extract_chain]
        N2[엔티티 스토어<br/>entity_store dict / DB]
        N3[엔티티 인식 체인<br/>entity_aware_chain]
    end

    OLD -- "LangChain 1.0.x<br/>재설계" --> NEW

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class E1,E2,E3 error
    class N1,N3 process
    class N2 storage
```

> `ConversationEntityMemory`와 `ConversationKGMemory`(그래프 메모리)는 LangChain 0.2.x 이후 deprecated 상태예요. 이 노트북에서 배울 Store + Tool 패턴으로 같은 기능을 더 유연하게 구현할 수 있어요.

## Modern 엔티티 메모리란?

이 노트북은 04번 노트북(`ConversationEntityMemory`)의 **Modern 버전**입니다. 같은 목표(대화에서 인물/역할 정보를 추출하고 기억하기)를 달성하지만, 접근 방식이 완전히 다릅니다.

### Legacy vs Modern 핵심 차이

| | Legacy (04번 노트북) | Modern (14번 노트북, 이 노트북) |
|---|---|---|
| 접근 방식 | `ConversationEntityMemory` 한 줄로 끝 | 추출 체인 + 스토어 + 포맷 함수로 분리 |
| 내부 동작 | 블랙박스 (클래스가 알아서 처리) | 투명 (각 단계를 직접 제어) |
| 커스터마이징 | 어려움 (클래스 상속 필요) | 쉬움 (함수만 교체하면 됨) |
| 저장소 | 클래스 내부 dict 고정 | dict, SQLite, Redis 등 자유롭게 교체 |

### 비유로 이해하기

- **Legacy** = "AI가 알아서 정리해주는 비서" (편하지만, 어떻게 정리하는지 모름)
- **Modern** = "내가 추출 규칙을 정의한 비서" (조금 더 손이 가지만, 모든 과정이 투명)

예를 들어:
```
Legacy:  memory = ConversationEntityMemory(llm=llm)  # 한 줄이면 끝!
         → 내부에서 엔티티를 어떻게 추출하고 저장하는지 알 수 없음

Modern:  1) entity_extract_chain  → 추출 규칙을 직접 작성
         2) entity_store (dict)    → 저장소를 직접 선택
         3) format_entity_summary  → 프롬프트 주입 방식을 직접 결정
         → 각 컴포넌트를 독립적으로 교체·확장 가능
```

### 왜 Modern 방식이 더 좋은가요?

1. **추출 규칙 커스터마이징**: 어떤 정보를 추출할지 프롬프트로 자유롭게 정의
2. **저장소 교체**: 인메모리 dict에서 PostgreSQL, Redis 등으로 쉽게 전환
3. **에이전트 연동**: LangGraph 에이전트의 도구(Tool)로 등록하여 자동 호출 가능
4. **디버깅 용이**: 각 단계의 입출력을 확인하고 문제를 쉽게 파악

In [1]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 세션별 대화 히스토리 (일반 대화용)
conversation_store: dict[str, ChatMessageHistory] = {}


def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]


# 세션별 엔티티 스토어 (아주 단순한 dict 기반 구조)
# 구조 예시: {"session_id": {"김철수": "프로젝트 A의 팀장" , ...}}
entity_store: dict[str, dict[str, str]] = {}


## 1. 엔티티 추출 체인 만들기

사용자 발화(utterance)에서 **인물·역할·프로젝트 정보를 요약한 텍스트**를 추출하는 체인을 만들어요. `entity_extract_chain`(엔티티 추출 체인)의 출력을 파싱해 세션별 딕셔너리에 저장하는 방식이에요.

실제 서비스에서는 JSON 구조로 파싱하는 것이 좋지만, 여기서는 개념 이해를 위해 사람이 읽기 좋은 한글 텍스트로만 정리해요.

### 엔티티 추출 흐름

```mermaid
flowchart TD
    UTTERANCE[사용자 발화] --> EXTRACT_PROMPT[entity_extract_prompt<br/>인물 / 역할 / 프로젝트 추출]
    EXTRACT_PROMPT --> LLM[ChatOpenAI]
    LLM --> SUMMARY[엔티티 요약 텍스트<br/>이름: 설명 형태]
    SUMMARY --> PARSE[줄 단위 파싱<br/>name, desc 분리]
    PARSE --> STORE[entity_store 갱신<br/>session_id 키로 저장]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class UTTERANCE input
    class EXTRACT_PROMPT,LLM,PARSE process
    class SUMMARY,STORE storage
```

> `entity_store`에는 **세션별로** 엔티티가 누적돼요. 같은 이름이 다시 등장하면 최신 정보로 덮어써요.

In [2]:
# ============================================================
# TODO: 엔티티 추출 프롬프트와 update_entity_store 함수를 구현하세요
# 힌트: LLM이 발화에서 "이름: 설명" 형식으로 엔티티를 추출하도록 프롬프트를 작성하세요
# 예상 결과: "김철수는 팀장이에요" 발화에서 {'김철수': '팀장'} 형태로 저장되어야 합니다
# ============================================================

# ---------------------------------------------------
# 엔티티 추출 체인 및 스토어 업데이트 함수 구현
# ---------------------------------------------------

# 1단계: 엔티티 추출용 프롬프트 정의
# 🎯 강의 포인트: Legacy의 ConversationEntityMemory가 내부적으로 하던 일을 명시적 함수로 구현합니다
# - 인물/역할/프로젝트 정보를 "이름: 설명" 형식으로 추출
entity_extract_prompt = ChatPromptTemplate.from_template(
    """
    너는 대화에서 인물/역할/프로젝트 정보를 정리하는 비서야.

    아래 발화에서 등장하는 인물과 그 역할, 참여 중인 프로젝트를
    한국어 한 줄 요약 목록으로 정리해줘.

    출력 형식 예시:
    - 김철수: 프로젝트 A 팀장, 백엔드 개발 담당
    - 박영희: 프로젝트 A 프론트엔드 개발자

    발화:
    {utterance}
    """
)

# entity_extract_chain: 발화 → 엔티티 요약 텍스트 생성
entity_extract_chain = entity_extract_prompt | model | StrOutputParser()


def update_entity_store(session_id: str, utterance: str) -> str:
    """주어진 발화에서 엔티티 요약을 추출하고, 세션별 엔티티 스토어를 업데이트.

    - 줄 단위로 파싱해 "이름: 설명" 패턴만 저장
    - 같은 이름이 다시 나오면 최신 정보로 덮어씀
    # 💡 팁: 실서비스에서는 pydantic 모델로 엔티티 스키마를 정의하고 JSON으로 파싱하면 더 안정적입니다
    """
    # 2단계: 엔티티 추출 체인 실행
    summary = entity_extract_chain.invoke({"utterance": utterance})

    if session_id not in entity_store:
        entity_store[session_id] = {}

    # 3단계: 결과를 줄 단위로 파싱해 스토어에 저장
    for line in summary.splitlines():
        line = line.strip("- ").strip()
        if not line or ":" not in line:
            continue
        # "이름: 설명" 형식 분리 (콜론 첫 번째 기준)
        name, desc = line.split(":", 1)
        entity_store[session_id][name.strip()] = desc.strip()

    return summary

## 2. 엔티티 메모리를 활용하는 답변 체인

세션별 엔티티 정보를 활용하여 **"누가 어떤 역할/프로젝트를 담당하는지"**를 기억하는 체인을 만들어요.

흐름:
1. 사용자 발화를 `ChatMessageHistory`(대화 이력 저장소)에 저장해요.
2. 같은 발화를 엔티티 추출 체인에 넣어 `entity_store`를 갱신해요.
3. 누적된 엔티티 정보를 `format_entity_summary()`로 텍스트화해 시스템 메시지로 LLM에 전달해요.

### 엔티티 메모리 체인 흐름

```mermaid
flowchart TD
    QUESTION[사용자 질문] --> HADD[history.add_user_message]
    QUESTION --> EXTRACT[update_entity_store<br/>엔티티 추출 + 저장]
    EXTRACT --> STORE[entity_store 갱신]
    STORE --> FMT[format_entity_summary<br/>엔티티 요약 텍스트]
    FMT --> PROMPT[entity_aware_prompt<br/>entity_summary + question]
    PROMPT --> LLM[ChatOpenAI]
    LLM --> HADD2[history.add_ai_message]
    HADD2 --> ANSWER[AI 응답]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef external fill:#e2e3e5,stroke:#6c757d,color:#383d41

    class QUESTION input
    class HADD,EXTRACT,FMT,PROMPT,LLM,HADD2 process
    class STORE storage
    class ANSWER output
```

> 엔티티 스토어는 **세션 내에서 계속 누적**돼요. 초기 발화에서 얻은 "김철수 = PM" 정보는 10번째 질문에서도 그대로 사용할 수 있어요.

In [3]:
# ============================================================
# TODO: 엔티티 정보를 활용하는 답변 체인(entity_memory_chain)을 구현하세요
# 힌트: 발화에서 엔티티를 추출·저장한 뒤, format_entity_summary()로 프롬프트에 주입하세요
# 예상 결과: "김철수가 어떤 역할이었죠?" 질문에 스토어의 정보로 정확히 답해야 합니다
# ============================================================

# ---------------------------------------------------
# 엔티티 인식 답변 체인 및 메모리 통합 함수
# ---------------------------------------------------

# 1단계: 엔티티 정보를 활용하는 답변 프롬프트
# - entity_summary: format_entity_summary()가 생성하는 텍스트
entity_aware_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
너는 프로젝트/팀 정보를 기억하는 PM 어시스턴트야.

아래는 이 세션에서 알고 있는 팀원/역할/프로젝트 정보야:
{entity_summary}

이 정보를 최대한 활용해서 사용자의 질문에 답변해 줘.
알 수 없는 정보는 추측하지 말고 모른다고 말해.
""",
        ),
        ("human", "{question}"),
    ]
)

# entity_aware_chain: 엔티티 요약을 시스템 메시지로 받아 답변 생성
entity_aware_chain = entity_aware_prompt | model | StrOutputParser()


def format_entity_summary(session_id: str) -> str:
    """세션의 엔티티 스토어를 LLM 프롬프트에 주입할 텍스트로 변환."""
    data = entity_store.get(session_id, {})
    if not data:
        return "(아직 저장된 엔티티 정보가 없습니다.)"
    # "- 이름: 설명" 형식으로 포맷
    lines = [f"- {name}: {desc}" for name, desc in data.items()]
    return "\n".join(lines)


def entity_memory_chain(inputs: dict) -> str:
    """엔티티 메모리를 활용하는 통합 체인 함수.

    실행 순서:
    1) 대화 히스토리에 사용자 발화 추가
    2) 발화에서 엔티티 추출 후 스토어 갱신
    3) 누적된 엔티티 정보를 텍스트로 변환해 프롬프트에 주입
    4) AI 응답을 대화 히스토리에 저장
    """
    session_id: str = inputs["session_id"]
    question: str = inputs["question"]

    # 1단계: 대화 히스토리에 사용자 발화 추가
    history = get_history(session_id)
    history.add_user_message(question)

    # 2단계: 발화에서 엔티티 추출 후 스토어 갱신
    _ = update_entity_store(session_id, question)

    # 3단계: 누적된 엔티티 정보를 텍스트로 변환해 프롬프트에 주입
    entity_summary = format_entity_summary(session_id)
    answer = entity_aware_chain.invoke(
        {
            "entity_summary": entity_summary,
            "question": question,
        }
    )

    # 4단계: AI 응답을 대화 히스토리에 저장
    history.add_ai_message(answer)
    return answer

In [4]:
# ---------------------------------------------------
# 엔티티 메모리 체인 테스트: 팀 구성 정보 누적 및 활용
# ---------------------------------------------------

session_id = "project_chat_01"

utterances = [
    "우리 팀에는 세 명이 있어요. 팀장은 김철수이고, 개발자는 박영희와 이민호입니다.",
    "김철수는 프로젝트 피닉스의 PM이고, 박영희는 프론트엔드, 이민호는 백엔드를 담당해요.",
    "이번 분기 안에 프로젝트 피닉스를 출시하는 것이 목표입니다.",
    "제가 팀 구성과 역할을 어떻게 설명드렸는지 요약해 주세요.",
]

print("=" * 60)
print("👥 엔티티 메모리 기반 프로젝트 대화")
print("=" * 60)

# 매 발화마다 엔티티가 추출·저장되고, 답변 시 활용됨
for i, q in enumerate(utterances, start=1):
    answer = entity_memory_chain({"session_id": session_id, "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:120]}...\n")

# 최종 엔티티 스토어 내용 확인
print("현재 엔티티 스토어 내용:")
print("-" * 60)
for name, desc in entity_store.get(session_id, {}).items():
    print(f"- {name}: {desc}")

👥 엔티티 메모리 기반 프로젝트 대화
1. [사용자] 우리 팀에는 세 명이 있어요. 팀장은 김철수이고, 개발자는 박영희와 이민호입니다.
   [AI] 네, 맞습니다. 팀에는 김철수 팀장과 두 명의 개발자, 박영희와 이민호가 있습니다. 추가로 궁금한 점이 있으신가요?...

2. [사용자] 김철수는 프로젝트 피닉스의 PM이고, 박영희는 프론트엔드, 이민호는 백엔드를 담당해요.
   [AI] 맞습니다! 김철수는 프로젝트 피닉스의 PM이고, 박영희는 프론트엔드 개발자, 이민호는 백엔드 개발자입니다. 이와 관련해 궁금한 점이 있으신가요?...

3. [사용자] 이번 분기 안에 프로젝트 피닉스를 출시하는 것이 목표입니다.
   [AI] 맞습니다! 프로젝트 피닉스는 이번 분기 출시 목표로 진행되고 있습니다. 김철수 PM과 박영희 프론트엔드 개발자, 이민호 백엔드 개발자가 함께 작업하고 있습니다. 추가로 궁금한 점이 있으신가요?...

4. [사용자] 제가 팀 구성과 역할을 어떻게 설명드렸는지 요약해 주세요.
   [AI] 팀 구성과 역할은 다음과 같습니다:

- 김철수: 프로젝트 피닉스의 PM (프로젝트 매니저)
- 박영희: 프로젝트 피닉스의 프론트엔드 개발자
- 이민호: 프로젝트 피닉스의 백엔드 개발자

또한, 프로젝트 피닉스는 이...

현재 엔티티 스토어 내용:
------------------------------------------------------------
- 김철수: 프로젝트 피닉스 PM
- 박영희: 프로젝트 피닉스 프론트엔드 개발자
- 이민호: 프로젝트 피닉스 백엔드 개발자
- 프로젝트 피닉스: 이번 분기 출시 목표


## 핵심 요약

### 이번 노트북의 구성 요소

| 구성 요소 | 역할 |
|----------|------|
| `entity_extract_chain` | 발화에서 인물·역할·프로젝트 정보를 텍스트로 추출 |
| `update_entity_store(session_id, utterance)` | 추출 결과를 파싱해 세션별 딕셔너리에 저장 |
| `format_entity_summary(session_id)` | 딕셔너리를 LLM 프롬프트에 주입할 텍스트로 변환 |
| `entity_aware_chain` | 엔티티 요약을 시스템 메시지로 받아 답변 생성 |
| `entity_memory_chain` | 위 과정을 한 번에 묶는 체인 함수 |

### 구식 vs Modern 패턴 설계 비교

| 항목 | 구식 `ConversationEntityMemory` | Modern Store + Tool |
|------|--------------------------------|---------------------|
| 추출 로직 | 클래스 내부에 고정 | `entity_extract_chain` 함수로 교체 가능 |
| 저장 구조 | 클래스 내부 dict | 외부 dict / SQLite / VectorStore로 교체 가능 |
| 조회 방식 | 클래스 메서드 | `format_entity_summary()` 함수 (자유롭게 확장) |
| LangGraph 연동 | 어려움 | Tool로 래핑해 에이전트에 바로 연결 가능 |

### 주의사항

- 엔티티 추출마다 LLM 호출이 발생해요. 호출 빈도 조절(예: 중요 발화만 추출)로 비용을 줄일 수 있어요.
- 이 노트북의 스토어는 dict 기반이에요. 실제 서비스에서는 `pydantic`으로 스키마를 정의하고 DB에 저장해요.
- 복잡한 인물 관계(A가 B의 상사이고, B는 C 프로젝트 담당)는 그래프 데이터베이스(Neo4j 등)가 더 적합해요.
- 같은 이름의 엔티티가 반복되면 최신 정보로 덮어쓰기 때문에, 정보 충돌이 발생할 수 있어요.

### 실전 확장 방향

- `pydantic` 모델로 엔티티 구조를 정의하고 JSON 파싱으로 정확도를 높여요.
- 벡터 저장소(vector store)에 엔티티를 저장하면 유사 검색이 가능해져요.
- LangGraph 에이전트에서 `entity_memory_chain`을 도구(tool)로 등록해 자동으로 호출할 수 있어요.

### 다음 단계

이 노트북으로 04번 Memory 섹션이 마무리돼요. 다음 섹션에서는 여러 체인을 연결하는 **멀티 체인 패턴**과 **에이전트(Agent)** 구조를 학습해요.

---

## 연습 문제

### 연습 1: 회의록 기반 엔티티 메모리 + 액션 아이템 추적

엔티티 추출 체인을 확장하여, 대화에서 **인물 정보뿐만 아니라 액션 아이템(할 일)도 추적**하는 회의 어시스턴트를 만들어 보세요.

**요구사항:**
1. 기존 `entity_store` 외에 **`action_store`** (딕셔너리)를 추가하여, 세션별로 액션 아이템을 저장하세요.
2. 액션 아이템 추출용 프롬프트를 만드세요 (예: "아래 발화에서 누가 무엇을 해야 하는지 추출해줘. 형식: `담당자: 할일`").
3. 아래 회의 시나리오로 테스트하세요:
   - "이번 프로젝트에 김대리, 박과장, 최사원이 참여합니다."
   - "김대리는 다음 주까지 시장 조사 보고서를 작성해야 합니다."
   - "박과장은 예산안을 금요일까지 검토하고, 최사원은 경쟁사 분석을 맡았습니다."
   - "지금까지 정리된 팀원 정보와 각자의 할 일을 알려주세요."
4. 마지막 질문에서 **엔티티 정보와 액션 아이템을 모두 활용**한 응답이 나오는지 확인하세요.

**힌트:**
- 기존 `entity_extract_chain`과 유사한 구조로 `action_extract_chain`을 만드세요.
- 프롬프트에 엔티티 요약과 액션 아이템 목록을 함께 넣어서 LLM이 종합적으로 답변하도록 구성하세요.
- `entity_memory_chain` 함수를 참고하여 엔티티 + 액션을 모두 업데이트하는 새 체인 함수를 만드세요.

In [5]:
# ============================================================
# TODO: 엔티티 + 액션 아이템을 함께 추적하는 회의 어시스턴트를 구현하세요
# 힌트: entity_store 외에 action_store를 추가하고, 각각 추출·저장 함수를 만드세요
# 예상 결과: 마지막 질문에서 팀원 정보와 할 일 목록을 함께 정리해야 합니다
# ============================================================

# ---------------------------------------------------
# 연습 1 풀이: 회의록 기반 엔티티 메모리 + 액션 아이템 추적
# ---------------------------------------------------

# 1단계: 액션 아이템 스토어 정의
# 구조: {"session_id": {"김대리": "다음 주까지 시장 조사 보고서 작성", ...}}
action_store: dict[str, dict[str, str]] = {}

# 2단계: 액션 아이템 추출 프롬프트 정의
action_extract_prompt = ChatPromptTemplate.from_template(
    """
    너는 회의 내용에서 액션 아이템(할 일)을 추출하는 비서야.

    아래 발화에서 누가 무엇을 해야 하는지 추출해줘.
    할 일이 없으면 "(할 일 없음)"이라고만 출력해.

    출력 형식 예시:
    - 김대리: 다음 주까지 시장 조사 보고서 작성
    - 박과장: 금요일까지 예산안 검토

    발화:
    {utterance}
    """
)

# action_extract_chain: 발화 → 액션 아이템 텍스트 생성
action_extract_chain = action_extract_prompt | model | StrOutputParser()


def update_action_store(session_id: str, utterance: str) -> str:
    """발화에서 액션 아이템을 추출하고 스토어에 저장."""
    result = action_extract_chain.invoke({"utterance": utterance})

    if session_id not in action_store:
        action_store[session_id] = {}

    # 줄 단위 파싱: "이름: 할일" 형식만 추출
    for line in result.splitlines():
        line = line.strip("- ").strip()
        if not line or ":" not in line or "할 일 없음" in line:
            continue
        name, action = line.split(":", 1)
        action_store[session_id][name.strip()] = action.strip()

    return result


def format_action_summary(session_id: str) -> str:
    """액션 아이템을 사람이 읽기 좋은 형태로 포맷."""
    data = action_store.get(session_id, {})
    if not data:
        return "(아직 등록된 액션 아이템이 없습니다.)"
    lines = [f"- {name}: {action}" for name, action in data.items()]
    return "\n".join(lines)


def meeting_assistant_chain(inputs: dict) -> str:
    """엔티티 + 액션 아이템을 모두 활용하는 회의 어시스턴트 체인."""
    session_id: str = inputs["session_id"]
    question: str = inputs["question"]

    # 1단계: 대화 히스토리 업데이트
    history = get_history(session_id)
    history.add_user_message(question)

    # 2단계: 엔티티 추출 & 스토어 업데이트 (기존 함수 재사용)
    _ = update_entity_store(session_id, question)

    # 3단계: 액션 아이템 추출 & 스토어 업데이트
    _ = update_action_store(session_id, question)

    # 4단계: 엔티티 + 액션 정보를 포함한 프롬프트 구성
    entity_summary = format_entity_summary(session_id)
    action_summary = format_action_summary(session_id)

    meeting_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "너는 회의를 정리하고 관리하는 PM 어시스턴트야.\n\n"
            "[팀원/역할 정보]\n{entity_summary}\n\n"
            "[액션 아이템 (할 일 목록)]\n{action_summary}\n\n"
            "위 정보를 최대한 활용해서 사용자의 질문에 답변해 줘.\n"
            "알 수 없는 정보는 추측하지 말고 모른다고 말해.",
        ),
        ("human", "{question}"),
    ])

    answer = meeting_prompt | model | StrOutputParser()
    result = answer.invoke({
        "entity_summary": entity_summary,
        "action_summary": action_summary,
        "question": question,
    })

    # 5단계: AI 응답 히스토리에 추가
    history.add_ai_message(result)
    return result


# 5단계: 회의 시나리오 테스트
meeting_session = "meeting_exercise_01"

meeting_utterances = [
    "이번 프로젝트에 김대리, 박과장, 최사원이 참여합니다.",
    "김대리는 다음 주까지 시장 조사 보고서를 작성해야 합니다.",
    "박과장은 예산안을 금요일까지 검토하고, 최사원은 경쟁사 분석을 맡았습니다.",
    "지금까지 정리된 팀원 정보와 각자의 할 일을 알려주세요.",
]

print("=" * 60)
print("[ 회의 어시스턴트 (엔티티 + 액션 아이템) ]")
print("=" * 60)

for i, q in enumerate(meeting_utterances, start=1):
    answer = meeting_assistant_chain({"session_id": meeting_session, "question": q})
    print(f"{i}. [사용자] {q}")
    print(f"   [AI] {answer[:200]}...\n")

# 스토어 내용 확인
print("=" * 60)
print("[ 엔티티 스토어 ]")
print("=" * 60)
for name, desc in entity_store.get(meeting_session, {}).items():
    print(f"  - {name}: {desc}")

print()
print("=" * 60)
print("[ 액션 아이템 스토어 ]")
print("=" * 60)
for name, action in action_store.get(meeting_session, {}).items():
    print(f"  - {name}: {action}")

[ 회의 어시스턴트 (엔티티 + 액션 아이템) ]
1. [사용자] 이번 프로젝트에 김대리, 박과장, 최사원이 참여합니다.
   [AI] 네, 이번 프로젝트에는 김대리, 박과장, 최사원이 참여합니다. 추가로 필요한 정보나 질문이 있으시면 말씀해 주세요!...

2. [사용자] 김대리는 다음 주까지 시장 조사 보고서를 작성해야 합니다.
   [AI] 맞습니다. 김대리는 다음 주까지 시장 조사 보고서를 작성하는 책임이 있습니다. 추가로 도움이 필요하시면 말씀해 주세요!...

3. [사용자] 박과장은 예산안을 금요일까지 검토하고, 최사원은 경쟁사 분석을 맡았습니다.
   [AI] 맞습니다. 박과장은 금요일까지 예산안을 검토해야 하고, 최사원은 경쟁사 분석을 맡고 있습니다. 추가로 필요한 정보나 도움이 필요하신가요?...

4. [사용자] 지금까지 정리된 팀원 정보와 각자의 할 일을 알려주세요.
   [AI] 현재 팀원 정보와 각자의 할 일은 다음과 같습니다.

**팀원 및 역할:**
- 김대리: 시장 조사 보고서 작성 담당
- 박과장: 예산안 검토 담당
- 최사원: 경쟁사 분석 담당

**액션 아이템 (할 일 목록):**
- 김대리: 다음 주까지 시장 조사 보고서 작성
- 박과장: 금요일까지 예산안 검토
- 최사원: 경쟁사 분석 맡기

추가적인 정보가 필요하시...

[ 엔티티 스토어 ]
  - 김대리: 시장 조사 보고서 작성 담당
  - 박과장: 예산안 검토 담당
  - 최사원: 경쟁사 분석 담당
  - 팀원 정보 요청: 팀원과 역할, 프로젝트에 대한 정보 정리 요청

[ 액션 아이템 스토어 ]
  - 김대리: 다음 주까지 시장 조사 보고서 작성
  - 박과장: 금요일까지 예산안 검토
  - 최사원: 경쟁사 분석 맡기


## 실전 예제: 엔티티 추출 기반 대화형 챗봇

엔티티 메모리 패턴을 활용한 **스타트업 팀 관리 챗봇**을 만들어 봅시다. 이 챗봇은:

1. 대화 중 언급되는 **팀원 정보를 자동으로 추출**합니다
2. 누적된 엔티티 정보를 활용해 **맥락 있는 답변**을 생성합니다
3. 4번 이상의 대화를 통해 엔티티가 **점진적으로 축적**되는 과정을 확인합니다
4. 최종적으로 **엔티티 스토어의 전체 내용**을 출력합니다

In [ ]:
# ---------------------------------------------------
# 실전 예제: 스타트업 팀 관리 챗봇
# ---------------------------------------------------
# 기존 entity_memory_chain을 재사용하여 스타트업 시나리오로 테스트합니다.
# 새 세션을 사용하므로 이전 테스트의 엔티티와 충돌하지 않습니다.

startup_session = "startup_team_01"

# 스타트업 팀 구성 대화 시나리오 (4턴 + 추가 질문)
startup_conversations = [
    "우리 스타트업 '테크노바'를 소개할게요. CEO는 정수진이고, CTO는 한동우입니다.",
    "정수진 CEO는 이전에 네이버에서 10년간 프로덕트 매니저로 일했어요. 한동우 CTO는 카카오 출신 시니어 개발자입니다.",
    "마케팅은 윤서연이 담당하고, 디자인은 프리랜서 강민재가 맡고 있어요. 윤서연은 인스타그램 마케팅 전문가예요.",
    "현재 시리즈 A 투자 유치를 준비 중이고, 목표 금액은 30억원입니다. 정수진 CEO가 투자 IR을 리드하고 있어요.",
]

print("=" * 60)
print("스타트업 팀 관리 챗봇 - 대화 시작")
print("=" * 60)

# 4번의 대화를 통해 엔티티가 점진적으로 축적됨
for i, q in enumerate(startup_conversations, start=1):
    answer = entity_memory_chain({"session_id": startup_session, "question": q})
    print(f"\n--- 턴 {i} ---")
    print(f"[사용자] {q}")
    print(f"[챗봇] {answer[:200]}...")

    # 매 턴마다 현재 엔티티 스토어 상태 출력
    current_entities = entity_store.get(startup_session, {})
    print(f"  [엔티티 수: {len(current_entities)}개]")

In [ ]:
# ---------------------------------------------------
# 엔티티 활용 확인: 축적된 정보를 바탕으로 질문하기
# ---------------------------------------------------

# 추가 질문으로 엔티티 메모리가 제대로 작동하는지 확인
followup_questions = [
    "CTO가 이전에 어디서 일했는지 알려줘.",
    "마케팅 담당자가 누구이고, 어떤 전문성이 있는지 정리해 줘.",
    "투자 IR을 누가 리드하고 있고, 목표 금액은 얼마인지 알려줘.",
    "테크노바 팀의 전체 구성을 표로 정리해 줘.",
]

print("=" * 60)
print("엔티티 활용 확인 - 추가 질문")
print("=" * 60)

for i, q in enumerate(followup_questions, start=1):
    answer = entity_memory_chain({"session_id": startup_session, "question": q})
    print(f"\n--- 추가 질문 {i} ---")
    print(f"[사용자] {q}")
    print(f"[챗봇] {answer[:300]}...")

In [ ]:
# ---------------------------------------------------
# 최종 엔티티 스토어 전체 내용 확인
# ---------------------------------------------------

print("=" * 60)
print("최종 엔티티 스토어 내용 (startup_team_01 세션)")
print("=" * 60)

final_entities = entity_store.get(startup_session, {})
print(f"총 {len(final_entities)}개의 엔티티가 저장되었습니다.\n")

for name, desc in final_entities.items():
    print(f"  - {name}: {desc}")

print("\n" + "=" * 60)
print("핵심 포인트:")
print("=" * 60)
print("1. 4번의 대화만으로 팀 전체의 정보가 자동으로 축적되었습니다.")
print("2. 추가 질문에서 엔티티 스토어의 정보를 활용해 정확히 답변했습니다.")
print("3. 같은 인물이 다시 언급되면 정보가 업데이트됩니다 (덮어쓰기).")
print("4. 이 패턴을 DB나 벡터 저장소와 연결하면 실서비스에 바로 적용 가능합니다.")